# 03 — Exploratory Data Analysis (EDA)
## Bluestock Mutual Fund Analytics Capstone — D3

**Objective:** Understand the data through distributions, trends, correlations,  
and category comparisons before building performance metrics.

Sections
--------
1. Universe Overview  
2. NAV Trend Analysis  
3. Return Distribution  
4. Volatility Analysis  
5. Correlation Matrix  
6. Category Comparison  
7. AMC Comparison  
8. Risk Level Analysis  
9. Seasonal Patterns  
10. Key EDA Insights


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']

BASE    = Path().resolve().parents[1]
DB_PATH = BASE / "datasets" / "db"       / "bluestock_mf.db"
ANA_CSV = BASE / "datasets" / "analytics"/ "analytics_ready.csv"
OUT_DIR = BASE / "datasets" / "analytics"
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(ANA_CSV, parse_dates=['date'])
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month

print(f"Loaded {len(df):,} rows | {df['scheme_code'].nunique()} funds | "
      f"{df['date'].min().date()} → {df['date'].max().date()}")


In [ ]:
# ── 1. UNIVERSE OVERVIEW ──────────────────────────────────────────────────────
with sqlite3.connect(DB_PATH) as conn:
    fm = pd.read_sql_query("""
        SELECT fm.scheme_code, fm.scheme_name, cm.category, cm.sub_category,
               fhm.amc_name, fm.risk_level, fm.benchmark,
               fm.expense_ratio, fm.fund_manager, fm.plan
        FROM fund_master fm
        LEFT JOIN category_master  cm  ON fm.category_id=cm.category_id
        LEFT JOIN fund_house_master fhm ON fm.amc_id=fhm.amc_id
    """, conn)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# By category
cat_counts = fm['sub_category'].value_counts()
axes[0].barh(cat_counts.index, cat_counts.values, color='steelblue')
axes[0].set_title('Funds by Sub-Category', fontweight='bold')
axes[0].set_xlabel('Count')

# By AMC
amc_counts = fm['amc_name'].value_counts()
axes[1].barh(amc_counts.index, amc_counts.values, color='darkorange')
axes[1].set_title('Funds by AMC', fontweight='bold')
axes[1].set_xlabel('Count')

# By Risk Level
risk_order = ['Low','Moderate','Moderately High','High','Very High']
risk_counts = fm['risk_level'].value_counts().reindex(
    [r for r in risk_order if r in fm['risk_level'].values])
risk_colors = ['green','limegreen','orange','darkorange','red'][:len(risk_counts)]
axes[2].bar(risk_counts.index, risk_counts.values, color=risk_colors)
axes[2].set_title('Funds by Risk Level', fontweight='bold')
axes[2].set_xlabel('Risk Level')
axes[2].set_ylabel('Count')
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_01_universe.png', dpi=130, bbox_inches='tight')
plt.show()
print("Fund universe breakdown saved")


In [ ]:
# ── 2. NAV TREND ANALYSIS ─────────────────────────────────────────────────────
# Normalise NAV to 100 at start date for fair comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

categories = ['Equity', 'Debt']
for ax, cat in zip(axes, categories):
    cat_funds = df[df['category'] == cat]['scheme_code'].unique()
    for i, code in enumerate(cat_funds):
        fund = df[df['scheme_code'] == code].sort_values('date')
        base_nav = fund['nav'].iloc[0]
        indexed  = fund['nav'] / base_nav * 100
        name     = fund['scheme_name'].iloc[0][:30]
        ax.plot(fund['date'], indexed, lw=1.2, label=name, alpha=0.85)
    ax.axhline(100, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.set_title(f'{cat} Funds — NAV Indexed to 100', fontweight='bold')
    ax.set_ylabel('Indexed NAV (Base=100)')
    ax.legend(fontsize=7, ncol=2, loc='upper left')

plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_02_nav_trend.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3. RETURN DISTRIBUTION ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# All-fund return histogram
all_ret = df['daily_return'].dropna()
axes[0,0].hist(all_ret.clip(-0.05, 0.05), bins=120,
               color='steelblue', edgecolor='none', alpha=0.8)
axes[0,0].axvline(0, color='black', lw=1, ls='--')
axes[0,0].set_title('Daily Return Distribution (All Funds)')
axes[0,0].set_xlabel('Daily Return')

# Box plot by category
cat_ret = df.groupby('category')['daily_return'].apply(list)
bp_data = [v for v in cat_ret.values]
bp = axes[0,1].boxplot(bp_data, labels=cat_ret.index, patch_artist=True,
                        medianprops=dict(color='black'))
for patch, color in zip(bp['boxes'], ['steelblue','darkorange','seagreen','purple']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0,1].set_title('Return Distribution by Category')
axes[0,1].set_ylabel('Daily Return')

# Annual return heatmap (pivot)
df['year_label'] = df['date'].dt.year
annual = df.groupby(['scheme_name','year_label'])['daily_return'].sum().unstack()
annual.index = [n[:20] for n in annual.index]
sns.heatmap(annual, ax=axes[1,0], cmap='RdYlGn', center=0,
            fmt='.2f', annot=True, annot_kws={'size':7}, linewidths=0.3)
axes[1,0].set_title('Cumulative Daily Returns by Year')
axes[1,0].set_xlabel('Year')
axes[1,0].tick_params(axis='x', labelsize=9)
axes[1,0].tick_params(axis='y', labelsize=7)

# Return stats table
stats = all_ret.describe().round(6)
axes[1,1].axis('off')
tbl = axes[1,1].table(
    cellText=[[f"{v:.6f}"] for v in stats.values],
    rowLabels=stats.index.tolist(), colLabels=['Value'],
    loc='center', cellLoc='left'
)
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
axes[1,1].set_title('Return Statistics', pad=20)

plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_03_return_dist.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4. VOLATILITY ANALYSIS ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rolling 90d vol by category
for cat, grp in df.groupby('sub_category'):
    avg_vol = grp.groupby('date')['rolling_90d_vol'].mean().dropna()
    if len(avg_vol) > 50:
        axes[0].plot(avg_vol.index, avg_vol.values * 100, lw=1.2, label=cat, alpha=0.85)
axes[0].set_title('Average 90-Day Rolling Volatility by Sub-Category')
axes[0].set_ylabel('Annualised Volatility %')
axes[0].legend(fontsize=8, ncol=2)

# Volatility by risk level (current)
latest = df.sort_values('date').groupby('scheme_code').last().reset_index()
risk_order = ['Low','Moderate','Moderately High','High','Very High']
risk_vol   = latest.groupby('risk_level')['rolling_90d_vol'].mean() * 100
risk_vol   = risk_vol.reindex([r for r in risk_order if r in risk_vol.index])
bar_colors = ['green','limegreen','orange','darkorange','red'][:len(risk_vol)]
axes[1].bar(risk_vol.index, risk_vol.values, color=bar_colors)
axes[1].set_title('Average Current Volatility by Risk Level')
axes[1].set_ylabel('Annualised Volatility %')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_04_volatility.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 5. CORRELATION MATRIX ─────────────────────────────────────────────────────
# Pivot daily returns — one column per fund
pivot = df.pivot_table(index='date', columns='scheme_name', values='daily_return')
pivot.columns = [c[:20] for c in pivot.columns]
corr = pivot.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, cmap='RdYlGn', center=0,
            vmin=-0.2, vmax=1.0, annot=True, fmt='.2f',
            annot_kws={'size': 6}, linewidths=0.3,
            xticklabels=True, yticklabels=True)
ax.set_title('Fund Return Correlation Matrix', fontsize=13, fontweight='bold')
ax.tick_params(axis='both', labelsize=7)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_05_correlation.png', dpi=130, bbox_inches='tight')
plt.show()

# Highest correlated pairs
corr_pairs = (corr.where(mask == False)
              .stack().reset_index()
              .rename(columns={0:'correlation',
                               'scheme_name':'fund_a',
                               'level_1':'fund_b'})
              .query('correlation < 1')
              .sort_values('correlation', ascending=False))
print("TOP 5 MOST CORRELATED FUND PAIRS:")
print(corr_pairs.head(5).to_string(index=False))
print()
print("TOP 5 LEAST CORRELATED FUND PAIRS:")
print(corr_pairs.tail(5).to_string(index=False))


In [ ]:
# ── 6. CATEGORY COMPARISON ────────────────────────────────────────────────────
# Load computed CAGR from performance_metrics
with sqlite3.connect(DB_PATH) as conn:
    pm = pd.read_sql_query("""
        SELECT pm.*, fm.scheme_name, cm.category, cm.sub_category, fhm.amc_name, fm.risk_level
        FROM performance_metrics pm
        JOIN fund_master fm ON pm.scheme_code=fm.scheme_code
        JOIN category_master cm ON fm.category_id=cm.category_id
        JOIN fund_house_master fhm ON fm.amc_id=fhm.amc_id
    """, conn)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# CAGR by sub-category (1Y)
pm_1y = pm[pm['period_label']=='1Y'].copy()
cat_cagr = pm_1y.groupby('sub_category')['cagr'].mean().sort_values(ascending=True) * 100
colors_bar = ['red' if v < 0 else 'steelblue' for v in cat_cagr.values]
axes[0].barh(cat_cagr.index, cat_cagr.values, color=colors_bar)
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Average 1Y CAGR by Sub-Category (%)', fontweight='bold')
axes[0].set_xlabel('CAGR %')

# Risk-Return scatter (1Y)
scatter_colors = {'Low':'green','Moderate':'limegreen',
                  'Moderately High':'orange','High':'darkorange','Very High':'red'}
for _, row in pm_1y.iterrows():
    color = scatter_colors.get(row['risk_level'], 'gray')
    axes[1].scatter(row['volatility_ann']*100, row['cagr']*100,
                    c=color, s=80, alpha=0.8, edgecolors='white', lw=0.5)

# legend
for label, color in scatter_colors.items():
    axes[1].scatter([], [], c=color, s=60, label=label)
axes[1].legend(title='Risk Level', fontsize=8)
axes[1].set_title('Risk vs Return (1Y, all funds)', fontweight='bold')
axes[1].set_xlabel('Annualised Volatility %')
axes[1].set_ylabel('1Y CAGR %')

plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_06_category.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7. AMC COMPARISON ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

amc_perf = pm_1y.groupby('amc_name').agg(
    avg_cagr=('cagr','mean'),
    avg_sharpe=('sharpe_ratio','mean'),
    fund_count=('scheme_code','count')
).reset_index().sort_values('avg_cagr', ascending=False)

x  = range(len(amc_perf))
w  = 0.4
b1 = ax.bar([i - w/2 for i in x], amc_perf['avg_cagr']*100, w,
            label='Avg 1Y CAGR %', color='steelblue', alpha=0.85)
ax2 = ax.twinx()
b2  = ax2.bar([i + w/2 for i in x], amc_perf['avg_sharpe'], w,
               label='Avg Sharpe', color='darkorange', alpha=0.75)

ax.set_xticks(list(x))
ax.set_xticklabels(amc_perf['amc_name'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Avg 1Y CAGR %', color='steelblue')
ax2.set_ylabel('Avg Sharpe Ratio', color='darkorange')
ax.set_title('AMC Comparison — Avg 1Y CAGR vs Sharpe Ratio', fontweight='bold')

lines = [b1, b2]
labels = [b.get_label() for b in lines]
ax.legend(lines, labels, loc='upper right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_07_amc.png', dpi=130, bbox_inches='tight')
plt.show()
print(amc_perf.to_string(index=False))


In [ ]:
# ── 8. RISK LEVEL ANALYSIS ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

risk_metrics = pm_1y.groupby('risk_level').agg(
    avg_cagr=('cagr','mean'),
    avg_vol=('volatility_ann','mean'),
    avg_mdd=('max_drawdown','mean'),
    avg_sharpe=('sharpe_ratio','mean'),
).reset_index()
risk_metrics['avg_cagr'] *= 100
risk_metrics['avg_vol']  *= 100
risk_metrics['avg_mdd']  *= 100
risk_order_map = {'Low':1,'Moderate':2,'Moderately High':3,'High':4,'Very High':5}
risk_metrics['order'] = risk_metrics['risk_level'].map(risk_order_map)
risk_metrics = risk_metrics.sort_values('order')

for ax, col, title, fmt in zip(
    axes,
    ['avg_cagr','avg_vol','avg_mdd'],
    ['Avg 1Y CAGR %','Avg Volatility %','Avg Max Drawdown %'],
    ['{:.1f}%','{:.1f}%','{:.1f}%']
):
    colors = ['green','limegreen','orange','darkorange','red'][:len(risk_metrics)]
    bars = ax.bar(risk_metrics['risk_level'], risk_metrics[col], color=colors)
    for bar, val in zip(bars, risk_metrics[col]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                fmt.format(val), ha='center', va='bottom', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Risk Level')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_08_risk.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 9. SEASONAL PATTERNS ──────────────────────────────────────────────────────
monthly = df.groupby(['month','category'])['daily_return'].mean().unstack() * 100
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly.index = [month_names[m] for m in monthly.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

monthly.plot(kind='bar', ax=axes[0], alpha=0.85, edgecolor='none')
axes[0].axhline(0, color='black', lw=0.7)
axes[0].set_title('Average Daily Return by Month & Category (%)', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Avg Daily Return %')
axes[0].legend(title='Category')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=0)

# Yearly avg returns (all equity funds)
equity_df = df[df['category']=='Equity'].copy()
yearly = equity_df.groupby(['year','scheme_name'])['daily_return'].sum().unstack() * 100
yearly.index = yearly.index.astype(str)
sns.heatmap(yearly.T, ax=axes[1], cmap='RdYlGn', center=0, annot=True,
            fmt='.1f', annot_kws={'size':8}, linewidths=0.5)
axes[1].set_title('Equity Fund Returns by Year (%)', fontweight='bold')
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_09_seasonal.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 10. KEY EDA INSIGHTS ──────────────────────────────────────────────────────
print("=" * 65)
print("KEY EDA INSIGHTS")
print("=" * 65)

# Best 1Y performer
best = pm_1y.sort_values('cagr', ascending=False).iloc[0]
worst = pm_1y.sort_values('cagr').iloc[0]
most_volatile = pm_1y.sort_values('volatility_ann', ascending=False).iloc[0]
best_sharpe   = pm_1y.sort_values('sharpe_ratio', ascending=False).iloc[0]
least_mdd     = pm_1y.sort_values('max_drawdown', ascending=False).iloc[0]

insights = [
    f"Best 1Y CAGR     : {best['scheme_name'][:40]} ({best['cagr']*100:.1f}%)",
    f"Worst 1Y CAGR    : {worst['scheme_name'][:40]} ({worst['cagr']*100:.1f}%)",
    f"Highest Volatility: {most_volatile['scheme_name'][:35]} ({most_volatile['volatility_ann']*100:.1f}%)",
    f"Best Sharpe Ratio : {best_sharpe['scheme_name'][:35]} ({best_sharpe['sharpe_ratio']:.3f})",
    f"Lowest Drawdown  : {least_mdd['scheme_name'][:40]} ({least_mdd['max_drawdown']*100:.1f}%)",
    f"Avg 1Y CAGR (Equity): {pm_1y[pm_1y['category']=='Equity']['cagr'].mean()*100:.1f}%",
    f"Avg 1Y CAGR (Debt)  : {pm_1y[pm_1y['category']=='Debt']['cagr'].mean()*100:.1f}%",
    f"Avg Correlation (all pairs): {corr_pairs['correlation'].mean():.3f}",
]
for i in insights:
    print(f"  {i}")

print()
print("  CHARTS SAVED TO: datasets/analytics/")
print("  eda_01_universe.png    — fund universe breakdown")
print("  eda_02_nav_trend.png   — indexed NAV trends")
print("  eda_03_return_dist.png — return distributions")
print("  eda_04_volatility.png  — rolling volatility")
print("  eda_05_correlation.png — correlation matrix")
print("  eda_06_category.png    — category comparison")
print("  eda_07_amc.png         — AMC comparison")
print("  eda_08_risk.png        — risk level analysis")
print("  eda_09_seasonal.png    — seasonal patterns")
print()
print("  ✅ EDA Complete — proceed to 04_performance_analytics.ipynb")
